## Data Generation for Monitoring System

This section generates a dataset of equipment operating parameters (cameras, temperature and vibration sensors). The data is stored in SQLite and Excel for subsequent analysis.

### Outline

1. Environment setup and library imports
2. Data generation
3. Anomaly injection
4. Export to SQLite and Excel

**Dataset specifications:**
- 5000 records
- 200 injected outliers
- Time span: 15.07.2009 – ~15.07.2012

In [1]:
import random as ra
import pandas as pd
import numpy as np
import datetime as dtime
import sqlite3


def generate_dates(start, passing, amount):
    current = start
    while amount >= 0:
        current = current + dtime.timedelta(days=passing)
        yield current
        amount -= 1


records, emissions = 5000, 200
# Random indices selected for anomaly injection
indexes = np.random.randint(0, records - 1, emissions)

# Valid value ranges for each parameter
limits = [[10, 11], range(160, 171), None,
          range(60, 71), list(range(1, 8)) + list(range(121, 131))]

### DataFrame Creation and Random Value Generation

In [2]:
data = pd.DataFrame(columns=['time', 'dynamic_range', 'viewing_angle',
                             'focal_length', 'temperature',
                             'oscillation_frequency'])

start_date = dtime.datetime(2009, 7, 15)
dates = []
for i in generate_dates(start_date, 1, records - 1):
    dates.append(i.strftime("%d.%m.%Y"))

data['time'] = dates
data['dynamic_range'] = np.random.randint(4, 9, records)
data['viewing_angle'] = np.random.randint(6, 160, records)
data['focal_length'] = np.random.uniform(2.8, 16, records)
data['temperature'] = np.random.randint(-10, 60, records)
data['oscillation_frequency'] = np.random.randint(8, 120, records)

### Anomaly Injection

Certain values are replaced with outliers to simulate equipment faults.

In [3]:
for col, limit in zip(data.columns[1:], limits):
    for i in indexes:
        try:
            # pick a random value from the defined limit set
            data.loc[i, col] = ra.choice(limit)
        except TypeError:
            # for continuous numeric ranges (when limit is None)
            data.loc[i, col] = ra.uniform(0.2, 2.8)

data.head(10)

,time,dynamic_range,viewing_angle,focal_length,temperature,oscillation_frequency
0,16.07.2009,7,15,3.395921,8,41
1,17.07.2009,4,88,7.910634,22,84
2,18.07.2009,4,79,7.070259,41,95
3,19.07.2009,5,33,6.621067,54,69
4,20.07.2009,6,90,12.573989,2,55
5,21.07.2009,5,84,3.460187,16,45
6,22.07.2009,7,56,12.495441,44,78
7,23.07.2009,6,149,12.323401,29,44
8,24.07.2009,4,43,8.174660,39,51
9,25.07.2009,7,71,4.940332,38,90


### Data Export

- SQLite (`readings` table)
- Excel (`database.xlsx`)

In [4]:
conn = sqlite3.connect('../data/database.db')
c = conn.cursor()
c.execute('''CREATE TABLE IF NOT EXISTS readings (time TIMESTAMP,
          dynamic_range INT, viewing_angle INT, focal_length NUMERIC,
          temperature INT, oscillation_frequency INT)''')
conn.commit()

data.to_sql('readings', conn, if_exists='replace', index=False)
data.to_excel('../data/database.xlsx', index=False)

conn.close()